In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import umap
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from torchvision import transforms
from PIL import Image
import glob
import joblib
from torch.utils.data import Dataset
from sklearn.metrics import mean_squared_error
from scipy.spatial.distance import pdist
import os

# =================== DATASET =======================
class CustomImageDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.image_paths = [f for f in glob.glob(os.path.join(data_dir, '**'), recursive=True)
                            if f.endswith('.png') or f.endswith('.jpg')]
        self.image_paths = self.image_paths[:10000]
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('L')
        if self.transform:
            image = self.transform(image)
        return image, img_path

# =================== LATENT VECTOR EXTRACTOR =======================
def extract_latent_vectors(model, dataloader, device):
    latent_vectors = []
    image_paths = []
    model.eval()
    with torch.no_grad():
        for imgs, paths in tqdm(dataloader, desc="Extracting latents"):
            imgs = imgs.to(device)
            _, mu, _ = model(imgs)  # VAE: recon, mu, logvar
            latent_vectors.append(mu.cpu())
            image_paths.extend(paths)
    return torch.cat(latent_vectors, dim=0), image_paths

# =================== UMAP APPROXIMATOR =======================
class SimpleUMAPApproximator(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64], output_dim=2):
        super().__init__()
        layers = []
        for h in hidden_dims:
            layers.append(nn.Linear(input_dim, h))
            layers.append(nn.ReLU())
            input_dim = h
        layers.append(nn.Linear(input_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x.view(x.size(0), -1))

def combined_loss(pred, target, alpha=0.5):
    mse = nn.MSELoss()(pred, target)
    d_pred = torch.cdist(pred, pred)
    d_target = torch.cdist(target, target)
    dist_loss = nn.MSELoss()(d_pred, d_target)
    return alpha * mse + (1 - alpha) * dist_loss

def improved_train_umap(latents, batch_size=32, epochs=100, device="cuda"):
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
    umap_targets = reducer.fit_transform(latents.cpu().numpy())
    umap_targets = torch.tensor(umap_targets, dtype=torch.float32)

    dataset = TensorDataset(latents, umap_targets)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = SimpleUMAPApproximator(latents.shape[1]).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = combined_loss(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1:03d} - Loss: {total_loss:.4f}")
    return model, reducer

# =================== VISUALIZATION =======================
def visualize_embeddings(original_umap, approximated_umap, title="UMAP Comparison"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    s1 = ax1.scatter(original_umap[:, 0], original_umap[:, 1], c=range(len(original_umap)), cmap='viridis', alpha=0.6)
    ax1.set_title("Original UMAP"); plt.colorbar(s1, ax=ax1)
    s2 = ax2.scatter(approximated_umap[:, 0], approximated_umap[:, 1], c=range(len(approximated_umap)), cmap='viridis', alpha=0.6)
    ax2.set_title("Neural UMAP"); plt.colorbar(s2, ax=ax2)
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# =================== MAIN =======================
def main_pipeline(model, dataset_path, batch_size=32, train_size=500):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.Grayscale(),
        transforms.ToTensor()
    ])
    dataset = CustomImageDataset(dataset_path, transform=transform)

    train_indices = list(range(train_size))
    test_indices = list(range(train_size, len(dataset)))

    train_loader = DataLoader(Subset(dataset, train_indices), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(Subset(dataset, test_indices), batch_size=batch_size, shuffle=False)

    print("→ Extracting train latents")
    latents, _ = extract_latent_vectors(model, train_loader, device)
    scaler = StandardScaler()
    latents_np = scaler.fit_transform(latents.numpy())
    latents = torch.tensor(latents_np, dtype=torch.float32)

    print("→ Training UMAP approximator")
    umap_model, reducer = improved_train_umap(latents.to(device), batch_size, 100, device=device)

    print("→ Evaluating on training data")
    with torch.no_grad():
        pred_latents = umap_model(latents.to(device)).cpu().numpy()
    orig_latents = reducer.embedding_
    visualize_embeddings(orig_latents, pred_latents)
    train_corr = np.corrcoef(pdist(orig_latents), pdist(pred_latents))[0, 1]

    print("→ Extracting test latents")
    test_latents, _ = extract_latent_vectors(model, test_loader, device)
    test_latents_np = scaler.transform(test_latents.numpy())
    test_latents = torch.tensor(test_latents_np, dtype=torch.float32)

    with torch.no_grad():
        pred_test = umap_model(test_latents.to(device)).cpu().numpy()
    orig_test = reducer.transform(test_latents_np)
    visualize_embeddings(orig_test, pred_test, "Test Set UMAP Comparison")

    test_corr = np.corrcoef(pdist(orig_test), pdist(pred_test))[0, 1]
    test_mse = mean_squared_error(orig_test, pred_test)

    print(f"✅ Train Corr: {train_corr:.4f}")
    print(f"✅ Test Corr:  {test_corr:.4f}")
    print(f"✅ Test MSE:   {test_mse:.6f}")
    save_dir = "umap_models"
    os.makedirs(save_dir, exist_ok=True)
    torch.save(umap_model.state_dict(), os.path.join(save_dir, "umap_approximator2.pth"))
    joblib.dump(reducer, os.path.join(save_dir, "umap_reducer2.pkl"))
    joblib.dump(scaler, os.path.join(save_dir, "scaler2.pkl"))

    print(f"✅ Models saved in '{save_dir}/'")

    return umap_model, reducer, train_corr, test_corr, test_mse

In [ ]:
# vae_model = VAE(...)   Loading vae model
vae_model.load_state_dict(torch.load("final_model.pth"))
main_pipeline(vae_model, "/pscratch/sd/m/monika/bl733_data_png/")